# freMTPL2: Understanding the Gold Standard of Actuarial Pricing Data

**Author:** Andres Gonzalez Ortega | **Date:** March 2026

---

## Why This Dataset Matters

The **freMTPL2** dataset is the actuarial profession's most widely-used benchmark for motor third-party liability (MTPL) pricing.
With **677,991 policies** from a French insurer, it has become the reference dataset in:
- CAS (Casualty Actuarial Society) pricing competitions
- Academic research on GLM vs ML pricing (Noll et al. 2018, Wuthrich 2019)
- Industry tutorials from the Swiss Association of Actuaries

### The Frequency-Severity Decomposition

Insurance pricing follows a fundamental decomposition:

**Pure Premium = E[Frequency] x E[Severity]**

- **Frequency**: How often does a policyholder file a claim? (Poisson-distributed)
- **Severity**: Given a claim, how large is it? (Gamma/Lognormal-distributed)

This notebook explores both components and the risk factors that drive them.

> *Nota para el contexto mexicano: En Mexico, el seguro de autos de terceros (Responsabilidad Civil) es obligatorio desde 2019. La CNSF (Comision Nacional de Seguros y Fianzas) regula las tarifas. Este tipo de analisis es directamente aplicable al mercado mexicano, aunque la estructura de datos de la AMIS (Asociacion Mexicana de Instituciones de Seguros) tiene variables distintas.*

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0] if Path().resolve().name == 'notebooks' else Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT.parent.parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns
import matplotlib.pyplot as plt

pio.templates.default = "plotly_white"
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Data Loading

We load the merged frequency-severity dataset via our data pipeline. The `load_merged()` function fetches from OpenML, cleans dtypes, aggregates severity per policy, and caches as parquet.

In [ ]:
from scripts.utils.data_loader import load_merged, split_data

df = load_merged()
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

## 2. First Look at the Data

In [ ]:
display(df.head(10))
print("\nData types:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

### So What?
Each row is a **policy-year**: one vehicle, one driver, one year of exposure. Key columns:
- `ClaimNb`: number of claims filed (target for frequency model)
- `Exposure`: fraction of year the policy was active (offset in Poisson model)
- `TotalClaimAmount` / `AvgClaimAmount`: severity targets
- `BonusMalus`: the driver's merit-demerit score (50 = best, up to 350)

> *En Mexico, el sistema de bonus-malus no esta tan estandarizado como en Francia. Cada aseguradora tiene su propio sistema de descuentos por buen historial. La CNSF ha propuesto estandarizarlo.*

## 3. Claim Rate and Exposure Distribution

In [ ]:
claim_rate = df['HasClaim'].mean()
total_claims = df['ClaimNb'].sum()
total_exposure = df['Exposure'].sum()
freq = total_claims / total_exposure

print(f"Policies with at least one claim: {claim_rate:.2%}")
print(f"Total claims: {total_claims:,}")
print(f"Total exposure (policy-years): {total_exposure:,.1f}")
print(f"Overall claim frequency: {freq:.4f} claims per policy-year")
print(f"\nClaim count distribution:")
print(df['ClaimNb'].value_counts().sort_index())

In [ ]:
fig = px.histogram(df, x='Exposure', nbins=50,
                   title='Exposure Distribution (Fraction of Year)',
                   labels={'Exposure': 'Exposure (years)', 'count': 'Number of Policies'},
                   color_discrete_sequence=['#2E86AB'])
fig.update_layout(showlegend=False)
fig.show()

### So What?
The claim rate is approximately **3.9%** -- meaning ~96% of policies have zero claims in a given year.
This extreme class imbalance is typical of insurance data and is precisely why we use Poisson regression
(which naturally handles the excess of zeros) rather than standard regression.

Most policies have full-year exposure (Exposure = 1.0), but there is a meaningful tail of partial-year
policies from mid-year starts or cancellations.

## 4. Numeric Feature Distributions

In [ ]:
numeric_features = ['VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'Density']

for feat in numeric_features:
    fig = px.histogram(df, x=feat, nbins=50,
                       title=f'Distribution of {feat}',
                       labels={feat: feat, 'count': 'Number of Policies'},
                       color_discrete_sequence=['#2E86AB'])
    fig.update_layout(showlegend=False)
    fig.show()
    print(f"{feat}: mean={df[feat].mean():.2f}, median={df[feat].median():.2f}, "
          f"std={df[feat].std():.2f}, min={df[feat].min()}, max={df[feat].max()}")
    print()

### So What?
Key observations:
- **VehPower**: Concentrated at 4-7, with a long right tail. Power rating correlates with vehicle value and speed capability.
- **VehAge**: Most vehicles are 0-10 years old. Very few exceed 20 years.
- **DrivAge**: Bimodal -- peaks around 30-40 and 50-60. Few very young drivers (unlike the US market).
- **BonusMalus**: Heavily concentrated near 50 (the minimum/best score). Most French drivers have good records.
- **Density**: Extremely right-skewed (Paris = 27,000 inhabitants/km2 vs rural areas < 50).

> *En Mexico, la flota vehicular es significativamente mas vieja en promedio (mediana ~12 anos vs ~6 en Francia). Esto afecta tanto la frecuencia como la severidad de los siniestros. Ademas, la densidad poblacional de la CDMX (~6,000 hab/km2) es comparable a las grandes ciudades francesas.*

## 5. Categorical Feature Distributions

In [ ]:
cat_features = ['VehBrand', 'VehGas', 'Area', 'Region']

for feat in cat_features:
    counts = df[feat].value_counts().head(20)
    fig = px.bar(x=counts.index.astype(str), y=counts.values,
                 title=f'Distribution of {feat} (Top 20)',
                 labels={'x': feat, 'y': 'Count'},
                 color_discrete_sequence=['#A23B72'])
    fig.update_layout(showlegend=False, xaxis_tickangle=-45)
    fig.show()

### So What?
- **VehBrand**: Brand B12 dominates (~30% of portfolio). Brands are anonymized but likely correspond to Renault, Peugeot, Citroen.
- **VehGas**: Roughly 50/50 split between Diesel and Regular (petrol). Diesel is much more common in Europe than in Mexico.
- **Area**: Areas are coded A (most rural) through F (most urban). Fairly even distribution.
- **Region**: 22 French regions (pre-2016 reorganization). R24 (Ile-de-France/Paris) is the largest.

## 6. Correlation Analysis

In [ ]:
corr_cols = ['ClaimNb', 'Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'Density',
             'HasClaim', 'TotalClaimAmount', 'PurePremium']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix -- Key Numeric Variables', fontsize=14)
plt.tight_layout()
plt.show()

### So What?
Correlations with `ClaimNb` are weak in absolute terms -- this is expected. Individual claims are stochastic
events; no single variable predicts them strongly. The art of pricing is combining many weak signals.

Notable patterns:
- **BonusMalus** has the strongest positive correlation with claims (~0.03) -- past behavior predicts future behavior.
- **Density** shows a positive correlation -- urban areas have more accidents.
- **VehAge** and **DrivAge** show weak negative correlations overall, but this hides non-linear effects we will explore next.

## 7. Claim Frequency by Driver Age -- The U-Shaped Curve

In [ ]:
# Compute claim frequency by driver age
age_freq = df.groupby('DrivAge').agg(
    total_claims=('ClaimNb', 'sum'),
    total_exposure=('Exposure', 'sum'),
    n_policies=('ClaimNb', 'count')
).reset_index()
age_freq['frequency'] = age_freq['total_claims'] / age_freq['total_exposure']

fig = px.line(age_freq, x='DrivAge', y='frequency',
              title='Claim Frequency by Driver Age -- The Classic U-Shaped Curve',
              labels={'DrivAge': 'Driver Age', 'frequency': 'Claim Frequency'})
fig.update_traces(line_color='#2E86AB', line_width=2)
fig.add_hline(y=age_freq['frequency'].mean(), line_dash='dash', line_color='gray',
              annotation_text='Portfolio Average')
fig.update_layout(xaxis_range=[18, 95])
fig.show()

### So What? -- A Key Actuarial Insight

The **U-shaped curve** is one of the most important empirical findings in motor insurance:
- **Young drivers (18-25)**: High frequency due to inexperience and risk-taking behavior
- **Middle-aged drivers (35-60)**: Lowest frequency -- experienced, cautious
- **Elderly drivers (70+)**: Frequency rises again due to declining reaction times and vision

This non-linear pattern is why linear models struggle with driver age. A simple coefficient cannot capture
the U-shape -- you need either **binning** (for GLMs) or **tree-based models** (which find it automatically).

> *En Mexico, la distribucion de edad de conductores es mas joven en promedio. Los conductores menores de 25 representan una proporcion mayor del mercado que en Francia, lo que amplifica el riesgo juvenil en la cartera.*

## 8. Claim Frequency Heatmap: Vehicle Age vs Driver Age

In [ ]:
# Bin ages for heatmap
df_heat = df.copy()
df_heat['DrivAge_bin'] = pd.cut(df['DrivAge'], bins=[17, 25, 35, 45, 55, 65, 75, 100],
                                labels=['18-25', '26-35', '36-45', '46-55', '56-65', '66-75', '76+'])
df_heat['VehAge_bin'] = pd.cut(df['VehAge'], bins=[-1, 1, 3, 5, 10, 15, 100],
                               labels=['0-1', '2-3', '4-5', '6-10', '11-15', '16+'])

# Compute frequency per cell
heat = df_heat.groupby(['DrivAge_bin', 'VehAge_bin']).agg(
    claims=('ClaimNb', 'sum'),
    exposure=('Exposure', 'sum')
).reset_index()
heat['frequency'] = heat['claims'] / heat['exposure']

pivot = heat.pivot(index='DrivAge_bin', columns='VehAge_bin', values='frequency')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Claim Frequency: Driver Age x Vehicle Age', fontsize=14)
ax.set_ylabel('Driver Age Group')
ax.set_xlabel('Vehicle Age Group')
plt.tight_layout()
plt.show()

### So What?
The heatmap reveals **interaction effects** that univariate analysis misses:
- **Young drivers + new cars**: Highest frequency cells. New car + inexperience = dangerous combination.
- **Old drivers + old cars**: Also elevated -- possibly reflects older vehicles with fewer safety features.
- The diagonal (age-matched driver and vehicle) is not uniformly risky -- the pattern is more complex.

These interactions are precisely what tree-based models capture that additive GLMs cannot (without explicit interaction terms).

## 9. BonusMalus: The Strongest Predictor

In [ ]:
# BonusMalus distribution
fig = px.histogram(df, x='BonusMalus', nbins=100,
                   title='BonusMalus Distribution',
                   labels={'BonusMalus': 'Bonus-Malus Coefficient', 'count': 'Policies'},
                   color_discrete_sequence=['#2E86AB'])
fig.show()

# BonusMalus vs claim frequency
bm_freq = df.groupby('BonusMalus').agg(
    claims=('ClaimNb', 'sum'),
    exposure=('Exposure', 'sum'),
).reset_index()
bm_freq['frequency'] = bm_freq['claims'] / bm_freq['exposure']
bm_freq = bm_freq[bm_freq['exposure'] > 10]  # filter noise

fig = px.scatter(bm_freq, x='BonusMalus', y='frequency', size='exposure',
                 title='Claim Frequency by BonusMalus -- Past Behavior Predicts Future Risk',
                 labels={'BonusMalus': 'Bonus-Malus Coefficient', 'frequency': 'Claim Frequency'})
fig.update_traces(marker_color='#A23B72')
fig.show()

### So What?
BonusMalus is the **single most predictive feature** for claim frequency. This makes intuitive sense:
it encodes the driver's entire claims history into a single number.

- Coefficient of 50 = best possible (no claims for many years) -- frequency ~0.03
- Coefficient of 150+ = poor history -- frequency >0.15 (5x higher)

The French bonus-malus system is regulated: -5% per claim-free year, +25% per at-fault claim.
This creates a natural credibility-weighted predictor.

> *En Mexico, las aseguradoras manejan descuentos por antiguedad sin siniestros (tipicamente 5-10% por ano hasta 40-50%), pero no hay un sistema estandarizado nacional como el frances. La propuesta seria adoptar un coeficiente similar para mejor discriminacion de riesgo.*

## 10. Severity Distribution -- The Heavy Right Tail

In [ ]:
# Filter to policies with claims
claims = df[df['TotalClaimAmount'] > 0].copy()

print(f"Policies with claims: {len(claims):,}")
print(f"Severity stats:")
print(claims['AvgClaimAmount'].describe())

fig = px.histogram(claims, x='AvgClaimAmount', nbins=100,
                   title='Claim Severity Distribution (Linear Scale)',
                   labels={'AvgClaimAmount': 'Average Claim Amount (EUR)', 'count': 'Count'},
                   color_discrete_sequence=['#D64933'])
fig.show()

fig = px.histogram(claims, x='AvgClaimAmount', nbins=100, log_x=True,
                   title='Claim Severity Distribution (Log Scale) -- Heavy Right Tail',
                   labels={'AvgClaimAmount': 'Average Claim Amount (EUR, log scale)', 'count': 'Count'},
                   color_discrete_sequence=['#D64933'])
fig.show()

print(f"\nPercentiles:")
for p in [50, 75, 90, 95, 99, 99.9]:
    print(f"  P{p}: EUR {claims['AvgClaimAmount'].quantile(p/100):,.0f}")

### So What?
Severity has the classic **heavy right tail** characteristic of insurance claims:
- Most claims are small (median ~EUR 1,200)
- But a few catastrophic claims exceed EUR 100,000+
- The mean is much higher than the median -- a hallmark of right-skewed distributions

This is why we use a **Gamma GLM** (or log-link model) for severity: it naturally handles positive,
right-skewed data. The log transformation linearizes the relationship.

The heavy tail also means that **reinsurance** and **large-loss capping** are important considerations
that this analysis does not fully address (but would in a production setting).

## 11. Exposure Analysis by Area

In [ ]:
area_stats = df.groupby('Area').agg(
    n_policies=('IDpol', 'count'),
    total_exposure=('Exposure', 'sum'),
    total_claims=('ClaimNb', 'sum'),
    avg_severity=('AvgClaimAmount', lambda x: x[x > 0].mean() if (x > 0).any() else 0),
).reset_index()
area_stats['frequency'] = area_stats['total_claims'] / area_stats['total_exposure']
area_stats['pct_portfolio'] = area_stats['n_policies'] / area_stats['n_policies'].sum() * 100

fig = px.bar(area_stats, x='Area', y=['frequency'],
             title='Claim Frequency by Area (A=Rural to F=Urban)',
             labels={'value': 'Claim Frequency', 'Area': 'Area Code'},
             color_discrete_sequence=['#2E86AB'])
fig.update_layout(showlegend=False)
fig.show()

display(area_stats.round(4))

### So What?
There is a clear **urban-rural gradient**: Area F (most urban, highest density) has approximately
2x the claim frequency of Area A (most rural). This makes sense: more traffic, more intersections,
more opportunities for collisions.

However, urban drivers also tend to have *lower severity* per claim (lower speeds, less catastrophic impacts).
The net effect on pure premium depends on which effect dominates.

> *En Mexico, la CDMX y area metropolitana concentran ~30% de la cartera de autos pero con frecuencia siniestral significativamente mayor. La diferencia rural-urbana es aun mas pronunciada que en Francia debido a la infraestructura vial.*

---

## Summary of EDA Findings

| Finding | Implication for Pricing |
|---------|----------------------|
| ~3.9% claim rate | Extreme class imbalance -- use Poisson, not logistic regression |
| U-shaped driver age curve | Need non-linear modeling (bins or trees) |
| BonusMalus dominates | Past claims history is the best predictor of future claims |
| Heavy-tailed severity | Use Gamma/Lognormal, not Normal distribution |
| Urban-rural gradient | Geographic rating is essential |
| Feature interactions | Tree models may outperform additive GLMs |

**Next notebook: 02_glm_models.ipynb** -- We build Poisson and Gamma GLMs as the regulatory baseline.